In [1]:
#The purpose of this script is to collect labelled data for 3 classifiers:
#1) An LSTM to recognise when a strikefoot is happening
#2) A Random Forest classifier to determine the strikefoot type (heel, midfoot, forefoot)
#3) A classifier to determine whether overstriding is occurring

#This script combines the functionality of strikefoot_type_trainer and data_extraction_1_2_3.
#A key limitation of data_extraction_1_2_3 was that the "u" frames were not saved. This fixes that.
import cv2
import numpy as np
import json
import sys
import pandas as pd
import torch
sys.path.append("/Users/abhinavarora/Desktop/CadenceCV/Scripts/Supplementary")
from numpy_encoder import NumpyEncoder
from video_extracter import extract_keypoints
from data_loader import training_df, testing_df
from torch import nn

In [ ]:
#Update per video
video_dir = "/Users/abhinavarora/Desktop/CadenceCV/Videos/Video10.mp4"
video_name = "Video10"

#Using extract_keypoints to obtain the essential keypoints from the video
coords = extract_keypoints(video_dir)

left_ankle_arr = coords['left_ankle']
right_ankle_arr = coords['right_ankle']
left_hip_arr = coords['left_hip']
right_hip_arr = coords['right_hip']
left_knee_arr = coords['left_knee']
right_knee_arr = coords['right_knee']
left_shoulder_arr = coords['left_shoulder']
right_shoulder_arr = coords['right_shoulder']
total_frames = coords['Frame Count']
n_detected = len(left_ankle_arr)
print(f'Keypoints detected in {n_detected}/{total_frames} frames')

#The purpose of this block is to extract the following 4 LSTM metrics across ALL frames:
#1) Velocity of the ankle x component
#2) Velocity of the ankle y component
#3) Velocity of the shin (x component of the vector between the knee and the ankle)
#4) Tibial angle
left_ankle_x_vel = np.gradient(left_ankle_arr[:, 0])
left_ankle_y_vel = np.gradient(left_ankle_arr[:, 1])
right_ankle_x_vel = np.gradient(right_ankle_arr[:, 0])
right_ankle_y_vel = np.gradient(right_ankle_arr[:, 1])

#Shin vector goes from ankle to knee
left_shin_vec = left_knee_arr - left_ankle_arr
right_shin_vec = right_knee_arr - right_ankle_arr
left_shin_velocity = np.gradient(left_shin_vec[:, 0])
right_shin_velocity = np.gradient(right_shin_vec[:, 0])

#Tibial angle: -> angle between the shin vector and the vertical (pi/2 subtracted since arctan2 is w.r.t the horizontal)
def _tibial(shin_vec_arr):
    return np.pi / 2 - np.arctan2(shin_vec_arr[:, 1], shin_vec_arr[:, 0])

left_tibial_angle = _tibial(left_shin_vec)
right_tibial_angle = _tibial(right_shin_vec)

#Draws a vertical line upward from the ankle — used to visually judge whether the foot
#lands in front of the pelvis (overstriding) or beneath it (neutral)
#extract_keypoints skips frames with no detection, so idx is clamped to the last detected frame
def draw_ankle_line(frame, actual_frame_idx, which_leg):
    idx = min(actual_frame_idx, n_detected - 1)
    ankle = left_ankle_arr[idx] if which_leg == 'left' else right_ankle_arr[idx]
    ax, ay = int(ankle[0]), int(ankle[1])
    if ax > 0:
        cv2.line(frame, (ax, ay), (ax, 0), (0, 0, 255), 2)
    return frame

#Calculates all 9 metrics at a given strikefoot frame for the given leg:
#LSTM metrics (4): ankle_x_vel, ankle_y_vel, shin_velocity, tibial_angle
#Classifier 1 metrics (5): knee flexion angle, ankle-hip horizontal offset,
#                           trunk angle, pre-strike ankle velocity, ankle approach angle
#All of these are measured at the strikefoot frame
def compute_all_metrics(actual_frame, which_leg):
    idx = min(actual_frame, n_detected - 1)
    if which_leg == 'left':
        ax_vel = left_ankle_x_vel[idx]
        ay_vel = left_ankle_y_vel[idx]
        sh_vel = left_shin_velocity[idx]
        tib = left_tibial_angle[idx]
        ankle_arr = left_ankle_arr
        knee_arr = left_knee_arr
        hip_arr = left_hip_arr
        shldr_arr = left_shoulder_arr
        all_ay = left_ankle_y_vel
    else:
        ax_vel = right_ankle_x_vel[idx]
        ay_vel = right_ankle_y_vel[idx]
        sh_vel = right_shin_velocity[idx]
        tib = right_tibial_angle[idx]
        ankle_arr = right_ankle_arr
        knee_arr = right_knee_arr
        hip_arr = right_hip_arr
        shldr_arr = right_shoulder_arr
        all_ay = right_ankle_y_vel

    #Knee flexion
    #np.arctan2 gives angles relative to the horizontal, so the angles must be subtracted
    v1 = hip_arr[idx] - knee_arr[idx]
    v2 = ankle_arr[idx] - knee_arr[idx]
    knee_flexion = float(np.rad2deg(np.arctan2(v2[1], v2[0]) - np.arctan2(v1[1], v1[0])) % 360)

    #Ankle-hip horizontal offset
    offset = float(np.abs(ankle_arr[idx, 0] - hip_arr[idx, 0]))

    #Trunk angle: -> angle between the shoulder and the hip w.r.t the vertical
    tv = shldr_arr[idx] - hip_arr[idx]
    trunk_angle = float(np.rad2deg(np.pi / 2 - np.arctan2(tv[1], tv[0])))

    #Pre-strikefoot ankle velocity (2 detected frames before the strikefoot)
    pre2 = max(0, idx - 2)
    ankle_vel_pre = float(all_ay[pre2])

    #Pre-strikefoot ankle movement direction angle
    p1 = max(0, idx - 1)
    p3 = max(0, idx - 3)
    dx = float(ankle_arr[p1, 0] - ankle_arr[p3, 0])
    dy = float(ankle_arr[p1, 1] - ankle_arr[p3, 1])
    approach_angle = float(np.arctan2(dy, dx))

    return {
        'ankle_x_vel': float(ax_vel),
        'ankle_y_vel': float(ay_vel),
        'shin_velocity': float(sh_vel),
        'tibial_angle': float(tib),
        'knee_flexion_angle': knee_flexion,
        'ankle_hip_horizontal_offset': offset,
        'trunk_angle': trunk_angle,
        'ankle_velocity_pre_strike': ankle_vel_pre,
        'ankle_approach_angle': approach_angle,
        'frame': idx,
        'side': which_leg
    }

In [ ]:
#This function will go through the video frame by frame. Each frame is classified as:
#"u" = the EXACT first frame where the foot begins going up (toe-off)
#"d" = the EXACT first frame where the foot contacts the ground (strikefoot)
#"s" = skip (if the frame is neither u nor d)
#"u" and "d" apply to the EXACT moment the foot goes up or down respectively.
#Each consecutive pair of "u" and "d" will be classified as a gait cycle.
def scrub_and_label(video_dir, which_leg):
    side_char = 'L' if which_leg == 'left' else 'R'
    cap = cv2.VideoCapture(video_dir)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    raw_labels = []
    current_u_frame = None

    for frame_idx in range(total):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break

        frame = draw_ankle_line(frame, frame_idx, which_leg)

        #Helps in identifying which foot is to be tracked
        h, w = frame.shape[:2]
        #Scale font size relative to frame width (1280 used as a reference baseline)
        scale = w / 1280
        fs = scale * 0.55
        fs_bottom = scale * 0.9
        th = max(1, int(scale * 2))
        mg = int(scale * 15)
        pink = (203, 192, 255)

        cv2.putText(frame, f'{which_leg.upper()} | Frame {frame_idx}',
                    (mg, mg + 24), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 0), th, cv2.LINE_AA)
        if current_u_frame is not None:
            cv2.putText(frame, f'u_frame={current_u_frame}',
                        (mg, mg + 50), cv2.FONT_HERSHEY_SIMPLEX, fs, (200, 200, 0), th, cv2.LINE_AA)
        cv2.putText(frame, 'u=up  d=down  s=skip  c=quit',
                    (mg, h - mg), cv2.FONT_HERSHEY_SIMPLEX, fs_bottom, pink, th, cv2.LINE_AA)

        cv2.imshow('Labelling', frame)
        key = cv2.waitKey(0) & 0xFF

        #Constructing a single gait cycle — u_frame is stored until the matching d completes it
        if key == ord('u'):
            current_u_frame = frame_idx

        elif key == ord('d'):
            #"d" is always the strikefoot frame — immediately prompt for strike type then overstriding
            strike_frame = frame.copy()
            #Input classification (h = heel, m = midfoot, f = forefoot, s = skip this strike entirely)
            cv2.putText(strike_frame, 'Strike: h=heel  m=midfoot  f=forefoot  s=skip',
                        (mg, h - mg), cv2.FONT_HERSHEY_SIMPLEX, fs_bottom, pink, th, cv2.LINE_AA)
            cv2.imshow('Labelling', strike_frame)
            pk = cv2.waitKey(0) & 0xFF

            if pk == ord('s'):
                current_u_frame = None
                continue

            strike_map = {ord('h'): 'heel', ord('m'): 'midfoot', ord('f'): 'forefoot'}
            if pk not in strike_map:
                current_u_frame = None
                continue
            strike_pattern = strike_map[pk]

            #Input overstriding classification — the vertical ankle line helps judge this
            #If the line falls anterior to the pelvis it is overstriding, if within it is neutral
            over_frame = frame.copy()
            cv2.putText(over_frame, 'Overstride: o=yes  n=neutral  s=skip',
                        (mg, h - mg), cv2.FONT_HERSHEY_SIMPLEX, fs_bottom, pink, th, cv2.LINE_AA)
            cv2.imshow('Labelling', over_frame)
            ok_key = cv2.waitKey(0) & 0xFF

            overstride_map = {ord('o'): 'overstriding', ord('n'): 'neutral', ord('s'): 'skip'}
            overstriding = overstride_map.get(ok_key, 'skip')

            #Appending the completed gait cycle and resetting u_frame for the next cycle
            raw_labels.append({
                'frame': frame_idx,
                'side': side_char,
                'strike_pattern': strike_pattern,
                'overstriding': overstriding,
                'u_frame': current_u_frame,
            })
            current_u_frame = None

        elif key == ord('c'):
            break

    cap.release()
    cv2.destroyAllWindows()
    return raw_labels

#These metrics must be extracted for ALL frames in each video before scrubbing begins
#Manual labelling done via tracking the left foot first
left_raw  = scrub_and_label(video_dir, 'left')
#This time, the right foot will be focused on to identify its gait cycles
right_raw = scrub_and_label(video_dir, 'right')

In [ ]:
#Obtaining metrics for each individual strikefoot frame and combining with the raw labels
#left and right entries are merged into a single list and sorted by frame number
all_entries = []

for raw in left_raw + right_raw:
    which_leg = 'left' if raw['side'] == 'L' else 'right'
    metrics = compute_all_metrics(raw['frame'], which_leg)
    entry = {
        'frame': raw['frame'],
        'side': raw['side'],
        'strike_pattern': raw['strike_pattern'],
        'overstriding': raw['overstriding'],
        'u_frame': raw['u_frame'],
        'video': video_name,
        **metrics,
    }
    all_entries.append(entry)

all_entries.sort(key=lambda x: (x['frame'], x['side']))
print(f'Total labeled strikes: {len(all_entries)}')

In [ ]:
#Data loading loop:
#1) Read existing data from strikefoot_data.json
#2) To the same list, add the new entries for this video
#3) Finally, rewrite strikefoot_data.json with the combined data
#If the file doesn't exist or holds old-format data (a dict), start fresh as an empty list
json_path = "/Users/abhinavarora/Desktop/CadenceCV/Json Data/strikefoot_data.json"
with open(json_path, 'r') as f:
    existing = json.load(f)

existing.extend(all_entries)
with open(json_path, 'w') as f:
    json.dump(existing, f, cls=NumpyEncoder, indent=4)


In [ ]:
video_extensions = [".mp4", ".MOV"]
base_video_dir_path = "/Users/abhinavarora/Desktop/CadenceCV/Videos"
vids = []
with open (json_path, "r") as file:
    data = json.load(file)

for frame in data:
    vids.append(frame["video"])


metadata = []
for video in vids:
    for ext in video_extensions:
        try:
            cap = cv2.VideoCapture(base_video_dir_path + "/" + video + ext)
            fps = cap.get(cv2.CAP_PROP_POS_FRAMES)
            #Gets width and heigt
            width  = cap.get(cv2.CV_CAP_PROP_FRAME_WIDTH)   
            height = cap.get(cv2.CV_CAP_PROP_FRAME_HEIGHT)
            print(width, height, fps)
            metadata.append({"Video": video, "Width": width, "Height": height, "FPS": fps})
            break
        except:
            pass

with open("/Users/abhinavarora/Desktop/CadenceCV/Json Data/video_meta_data.json", "w") as file:
    json.dump(metadata, file, indent=4, cls=NumpyEncoder)


In [ ]:
#Dropping the rows where the u_frame doesn't exist (That is the only thing that doesn't exist)
training_df = training_df.dropna(axis=0, how='any')
#Obtaining the 4 metrics required for footnet: ankle_x_vel, tibial_angle, shin_velocity, ankle_y_vel
footnet_metrics = ["ankle_x_vel", "tibial_angle", "shin_velocity", "ankle_y_vel", "u_frame", "video", "frame", "side"]
#First step: Separate the training_df by video into different dfs
videos = set()
for index, value in training_df["video"].items():
    videos.add(value)

dfs_to_merge = []
for v in videos:
    df = pd.DataFrame(training_df[training_df["video"] == v])
    df = df[footnet_metrics]
    #Now, sorting items by leg, then frame. This is important to label contact and non-contact strikes effectively
    df.sort_values(by=["side", "frame"], ascending=True, inplace=True)
    dfs_to_merge.append(df)

footnet_training_df = pd.DataFrame(pd.concat(dfs_to_merge, axis=0))


In [ ]:
#Now, obtaining the data from frame_by_frame_data.json
with open("/Users/abhinavarora/Desktop/CadenceCV/Json Data/frame_by_frame_data.json", "r") as file:
    data = json.load(file)

video_metric_data = []
for key in data:
    df = pd.DataFrame(data[key])
    video_metric_data.append(df)

video_metric_df = pd.DataFrame(pd.concat(video_metric_data, axis=0))

#Dropping with any NA values
video_metric_df = video_metric_df.dropna(axis=0, how='any')

video_metric_df.loc[video_metric_df.side == "left", "side"] = "L"
video_metric_df.loc[video_metric_df.side== "right", "side"] = "R"

In [ ]:
#Now, adding a column to video_metric_df to indicate where there is contact and no contact between the ground
#Non-contact points: each row in footnet_training_df's u and d frames (between those frames)
#Contact points: done via obtaining the previous row's d frame and the current row's u frame
footnet_training_df.reset_index(drop=True)
#Initialsing label values to -1 ->Indicates there isn't enough info to fill in contact or no contact
video_metric_df["label"] = -1


In [ ]:
# Reset index first to ensure iloc works as expected
footnet_training_df = footnet_training_df.reset_index(drop=True)
video_metric_df = video_metric_df.copy()

for i, (index, row) in enumerate(footnet_training_df.iterrows()):
    # Non-contact points
    video_metric_df.loc[
        (video_metric_df["video"] == row["video"]) & 
        (video_metric_df["frame"] >= row["u_frame"]) & 
        (video_metric_df["frame"] < row["frame"]) &
        (video_metric_df["side"] == row["side"]), 
        "label"
    ] = 0
    
    # Contact points
    if i > 0:
        prev_row = footnet_training_df.iloc[i - 1]
        if prev_row["video"] == row["video"] and prev_row["side"] == row["side"]:
            video_metric_df.loc[
                (video_metric_df["video"] == row["video"]) & 
                (video_metric_df["frame"] >= prev_row["frame"]) & 
                (video_metric_df["frame"] < row["u_frame"]) &
                (video_metric_df["side"] == row["side"]),
                "label"
            ] = 1

In [ ]:
for i, (index, row) in enumerate(footnet_training_df.iterrows()):
    if i > 0:
        prev_row = footnet_training_df.iloc[i-1]
        if prev_row["video"] == row["video"] and prev_row["side"] == row["side"]:
            contact_length = row["u_frame"] - prev_row["frame"]
            non_contact_length = row["frame"] - row["u_frame"]
            print(f"Contact frames: {contact_length}, Non-contact frames: {non_contact_length}")
            break

In [ ]:
#First step: slicing the frame_by_frame data accordingly tp obtain the non-contact points 
nocontact_points_dfs = []
footnet_training_df.reset_index(drop=True)
for i, (index, row) in enumerate(footnet_training_df.iterrows()):
    df = video_metric_df[(video_metric_df["video"] == row["video"]) & (video_metric_df["frame"] >= row["u_frame"]) & (video_metric_df["frame"] <= row["frame"]) & (video_metric_df["side"] == row["side"])]
    #Filling the columns up with 1s to label them as contact points (using 1s and 0s)
    df["label"] = 0
    nocontact_points_dfs.append(df)

#Second step: obtaning the frames where there is a contact point. This is done via obtaining the previous row's d frame and the current row's u frame
gait_cycle_dfs = []
footnet_training_df.reset_index(drop=True)
for i, (index, row) in enumerate(footnet_training_df.iterrows()):
    if index > 0:
        df = video_metric_df[(video_metric_df["video"] == row["video"]) & (video_metric_df["frame"] >= footnet_training_df.iloc[i-1]["frame"]) & (video_metric_df["frame"] <= row["u_frame"])]
        df["label"] = 1
        gait_cycle_dfs.append(df)

In [ ]:
non_contact_points_df = pd.concat(nocontact_points_dfs, axis=0)
gait_cycle_df = pd.concat(gait_cycle_dfs, axis=0)
gait_cycle_df.reset_index(drop=True)
non_contact_points_df.reset_index(drop=True)


In [ ]:
video_metric_df[]

In [ ]:
#Finally constructing the LSTM to detect strikefoot 
#Input would be a tensor with shape: 
#(batch size (an X number of gait cycles will be fed at a time), A sequence length (the no. of hidden states), Input size = number of features there exist)
#Hidden size is the length of each hidden state vector

lstm = nn.LSTM(input_size=4, hidden_size=32, num_layers=1, batch_first=True, bidirectional=True)
#Input will have the following shape (32, 50, 4) for 32 gait cycles, 50 timesteps, 4 input features

